# Speech Emotion Recognition — TESS

Classify 7 emotions from speech audio using MFCC features and an LSTM.

**Dataset:** Toronto Emotional Speech Set (TESS) — 2,800 audio files, 2 actresses × 7 emotions × 200 utterances.
- Kaggle: https://www.kaggle.com/datasets/ejlok1/toronto-emotional-speech-set-tess
- Original: https://tspace.library.utoronto.ca/handle/1807/24487

**Pipeline:** load WAV → extract MFCC (40 coefficients, time dimension preserved) → LSTM → Dense → Softmax.

Designed to run on Kaggle with a GPU accelerator (~5 min end-to-end). See README for setup.

## 1. Setup

In [ ]:
# Kaggle's base image already ships librosa — no install needed.
import librosa
print('librosa version:', librosa.__version__)

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TF version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Attach the dataset

In the Kaggle notebook editor: **Add Input** (right sidebar) → search *toronto emotional speech set* → add `ejlok1/toronto-emotional-speech-set-tess`. It mounts read-only at `/kaggle/input/toronto-emotional-speech-set-tess/`.

Also: **Settings → Accelerator → GPU T4 ×2** (or any GPU) and **Internet → On** (only needed if pip-installing). Then run the cell below to confirm the dataset is there.

In [ ]:
!ls /kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess | head

## 3. Build a file index

TESS filenames look like `OAF_back_angry.wav` or `YAF_dog_ps.wav` — the emotion is the last underscore-separated token (`ps` = pleasant surprise).

In [ ]:
DATA_ROOT = '/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess'

def build_index(root):
    records = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if not fn.lower().endswith('.wav'):
                continue
            tokens = fn.lower().replace('.wav', '').split('_')
            emotion = tokens[-1]
            if emotion == 'ps':
                emotion = 'surprise'
            records.append({'path': os.path.join(dirpath, fn), 'emotion': emotion})
    return pd.DataFrame(records)

df = build_index(DATA_ROOT)
print(f'Total files: {len(df)}')
print(df['emotion'].value_counts())
df.head()

In [ ]:
# The Kaggle mirror duplicates the folder structure (5600 files = 2800 × 2).
# Dedupe by filename so train/test don't see the same audio twice.
df['filename'] = df['path'].apply(os.path.basename)
df = df.drop_duplicates(subset='filename').drop(columns='filename').reset_index(drop=True)
print(f'Total files (after dedup): {len(df)}')
print(df['emotion'].value_counts())

## 4. Inspect a sample

In [ ]:
sample = df.sample(1, random_state=SEED).iloc[0]
y_demo, sr_demo = librosa.load(sample['path'], sr=22050)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
librosa.display.waveshow(y_demo, sr=sr_demo, ax=axes[0])
axes[0].set_title(f"Waveform — {sample['emotion']}")

mel = librosa.feature.melspectrogram(y=y_demo, sr=sr_demo, n_mels=128)
mel_db = librosa.power_to_db(mel, ref=np.max)
img = librosa.display.specshow(mel_db, sr=sr_demo, x_axis='time', y_axis='mel', ax=axes[1])
axes[1].set_title('Mel spectrogram')
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
plt.tight_layout(); plt.show()

## 5. Feature extraction — MFCC (time dimension preserved)

Each clip becomes a tensor of shape `(n_frames, n_mfcc)`. The LSTM consumes the frame axis as its sequence dimension.

In [ ]:
SAMPLE_RATE = 22050
DURATION    = 3.0           # seconds; TESS clips are ~2 s, so we pad/trim
N_MFCC      = 40
N_SAMPLES   = int(SAMPLE_RATE * DURATION)

def extract_mfcc(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    else:
        y = y[:N_SAMPLES]
    mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    return mfcc.T.astype(np.float32)   # (n_frames, n_mfcc)

features = [extract_mfcc(p) for p in tqdm(df['path'].values, desc='Extracting MFCC')]
X = np.stack(features)
print('Feature tensor shape:', X.shape)

## 6. Encode labels and split (stratified train / val / test)

In [ ]:
le = LabelEncoder()
y_int = le.fit_transform(df['emotion'].values)
y_cat = to_categorical(y_int)
classes = le.classes_
print('Classes:', list(classes))

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_cat, test_size=0.30, random_state=SEED, stratify=y_int
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED,
    stratify=np.argmax(y_temp, axis=1)
)
print(f'Train {X_train.shape}  Val {X_val.shape}  Test {X_test.shape}')

# Standardize using training stats only
mean = X_train.mean(axis=(0, 1))
std  = X_train.std(axis=(0, 1)) + 1e-9
X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std

## 7. Model

In [ ]:
n_frames, n_features = X_train.shape[1], X_train.shape[2]
n_classes = y_cat.shape[1]

model = Sequential([
    LSTM(256, input_shape=(n_frames, n_features), return_sequences=False),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(n_classes, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 8. Train

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
    ModelCheckpoint('/kaggle/working/best_model.keras', monitor='val_accuracy', save_best_only=True),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout(); plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Evaluate on the held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=classes))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save model and preprocessing stats

In [ ]:
model.save('/kaggle/working/emotion_lstm.keras')
np.savez('/kaggle/working/preprocess.npz', mean=mean, std=std, classes=classes)
print('Saved emotion_lstm.keras and preprocess.npz to /kaggle/working/')

## 11. Inference helper

In [ ]:
def predict_emotion(audio_path, model, mean, std, classes):
    feats = extract_mfcc(audio_path)
    feats = (feats - mean) / std
    feats = np.expand_dims(feats, axis=0)
    probs = model.predict(feats, verbose=0)[0]
    idx = int(np.argmax(probs))
    return classes[idx], float(probs[idx]), dict(zip(classes, probs.tolist()))

# Demo on a random file
demo = df.sample(1, random_state=99).iloc[0]
pred, conf, all_probs = predict_emotion(demo['path'], model, mean, std, classes)
print(f"File:      {demo['path']}")
print(f"True:      {demo['emotion']}")
print(f"Predicted: {pred}  (confidence {conf:.2%})")